In [1]:
# imports
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch

c:\PROJECTS\ai_ml\NLP\nlp-sample_codes\unit6-exercise\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset

# loading the dataset directly from the downloaded text file
dataset = load_dataset(
    "csv", 
    data_files="financial_phrasebank/Sentences_AllAgree.txt", 
    delimiter="@", 
    column_names=["sentence", "label"], 
    encoding="latin1"
)

# convert string labels ('neutral', 'positive', 'negative') to integers for the model
label2id = {"negative": 0, "neutral": 1, "positive": 2}
dataset = dataset.map(lambda x: {"label": label2id[x["label"]]})

# splitting the dataset
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(300))
dataset["test"] = dataset["test"].select(range(100))

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2037
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 227
    })
})


In [3]:
# tokenizer initialization
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 227/227 [00:00<00:00, 2357.90 examples/s]


In [4]:
# creating the model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3222.94it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    accuracy = accuracy_score(labels, preds)
    
    # Return a proper dictionary
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# training configuration
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no"
)

In [7]:
# model trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].shuffle(seed=42),
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

print("Training started...")
trainer.train()

Training started...


c:\PROJECTS\ai_ml\NLP\nlp-sample_codes\unit6-exercise\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# model evaluation
print("\nEvaluating...")
results = trainer.evaluate()
print(results)

In [ ]:
# PCA visualization
print("\nGenerating PCA...")

words = [
    "profit", "loss", "revenue", "growth", "decline",
    "stocks", "shares", "market", "investment", "dividend",
    "bullish", "bearish", "inflation", "economy", "trade",
    "bank", "loan", "credit", "risk", "return"
]

embeddings = []

for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = model.distilbert(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).numpy()
    embeddings.append(embedding[0])

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(8,6))
for i, word in enumerate(words):
    x, y = reduced[i]
    plt.scatter(x, y)
    plt.text(x+0.01, y+0.01, word)

plt.title("PCA of Financial Word Embeddings (Fine-tuned DistilBERT)")
plt.savefig("pca_plot.png")
plt.show()